# Dynamic Prompts (`@dynamic_prompt`)

A **dynamic prompt** lets you build the system prompt *at runtime* instead of hard-coding one string. The `@dynamic_prompt` middleware runs right before each model call and returns the system prompt to use for that call.

```
request (state + runtime context) --> @dynamic_prompt --> system prompt --> model
```

Typical uses: personalise by user, switch language, change persona by tier, or inject the current date / live data.

In [1]:
%run ../../langchain_common.py

C:\Users\akhawaja\git\cs4603\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


USER_AGENT environment variable not set, consider setting it to identify your requests.


## Personalise the prompt by customer tier

We pass a typed **runtime context** (`SupportContext`) when invoking the agent. The dynamic prompt reads that context and tailors the system prompt for each request.

In [2]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest


@dataclass
class SupportContext:
    user_name: str
    tier: str  # "free" or "premium"


@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    ctx = request.runtime.context
    base = "You are a customer-support assistant for an online store."
    if ctx.tier == "premium":
        return (
            f"{base} You are speaking with {ctx.user_name}, a PREMIUM customer. "
            "Be warm, thorough, and proactively offer extra help."
        )
    return (
        f"{base} You are speaking with {ctx.user_name}, a free-tier customer. "
        "Be polite but concise."
    )


agent = create_agent(
    model=llm_noreason,
    tools=[],
    context_schema=SupportContext,
    middleware=[personalized_prompt],
)

### Premium customer
Same question, but the prompt is built for a premium user.

In [3]:
resp = agent.invoke(
    {"messages": [HumanMessage(content="My order is late. What can you do?")]},
    context=SupportContext(user_name="Ada", tier="premium"),
)
resp["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hello Ada! It is so wonderful to hear from you. First and foremost, please accept my sincere apologies that your order hasn't arrived as expected. As one of our most valued **PREMIUM members**, your time is incredibly important to us, and we know how frustrating it is when things don't go according to plan.

I want to make this right for you immediately. Here is exactly what I can do to help:

1.  **Immediate Investigation**: I can pull up your order details right now to see exactly where the package is stuck in the logistics chain. Was it a carrier delay, a warehouse hold-up, or perhaps a customs issue?
2.  **Expedited Resolution**: If the package is significantly delayed, I can personally contact our logistics partner to prioritize your shipment or, if necessary, arrange for an **overnight replacement** to be shipped to you at no cost.
3.  **Premium Compensation**: Since you are a PREMIUM customer, I'd 

### Free-tier customer
The exact same code path produces a different, more concise persona.

In [4]:
resp = agent.invoke(
    {"messages": [HumanMessage(content="My order is late. What can you do?")]},
    context=SupportContext(user_name="Sam", tier="free"),
)
resp["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hello Sam, I'm sorry to hear your order is running late.

Since you are on our free tier, I can't offer a refund or expedited shipping, but I can:
1. **Check the current status** of your shipment immediately.
2. **Contact the carrier** to investigate the delay.
3. **Provide a tracking link** if you don't have one handy.

Could you please share your **order number** so I can look into this for you?


## Summary

- `@dynamic_prompt` returns a **string** (or `SystemMessage`) used as the system prompt for that model call.
- It receives a `ModelRequest`, so it can read `request.runtime.context` and `request.state`.
- Great for personalisation, localisation, persona switching, and injecting live data such as the current date.